*Please name your Example notebook as transparently as possible. Feel free to add a description of the workflow's goal. [Remove this line when creating your example]*

# Lesion segmentation with LINDA

**Author of this template** : Michèle Masson-Trottier [Remove this line when creating your example]

**Author**: 
Michèle Masson-Trottier

### Citation:

#### Tools included in this workflow
Pustina, D., Coslett, H. B., Turkeltaub, P. E., Tustison, N., Schwartz, M. F., & Avants, B. (2016). Automated segmentation of chronic stroke lesions using LINDA: Lesion identification with neighborhood data analysis. Hum Brain Mapp, 37(4), 1405-1421. https://doi.org/10.1002/hbm.23110 

#### Publications
*Any publication that helped you build your example should be cited here*

#### Educational resources
*Any educational resource (for example Andy's Brain Book, FSL tutorials...) should be cited here.*

#### Dataset
Makayla Gibson, Roger Newman-Norlund, Leonardo Bonilha, Julius Fridriksson, Gregory Hickok, Argye E. Hillis, Dirk-Bart den Ouden, and Chris Rorden (2024). Aphasia Recovery Cohort (ARC) Dataset. OpenNeuro. [Dataset] doi: doi:10.18112/openneuro.ds004884.v1.0.2

## Load software tools and import python libraries

Please have a bloc for loading the different modules needed for your example. If you can not find all the required tools in Neurodesk, feel free to add tools using the web-based GUI https://www.neurodesk.org/neurocontainers-ui/

In [1]:
#load LINDA
import module
await module.load('linda/0.5.1')
await module.list()

['linda/0.5.1']

In [2]:
%%capture 
!pip install pydicom mystmd nilearn

In [3]:
# Import the necessary libraries
import pydicom # Library to work with DICOM files
import matplotlib.pyplot as plt # Plotting library for displaying images
import os # Standard library to interact with the filesystem
from pathlib import Path
import warnings
import nibabel as nib
from nilearn import datasets

In [4]:
# ------------------------------------------------------------
# Notebook paths (single source of truth)
# ------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()          # where you launched the notebook
PROJECT_DIR  = NOTEBOOK_DIR        # change if you want a different root

# Data locations
RAW_DATASET = PROJECT_DIR / "ds004884"
DEST        = PROJECT_DIR / "ds004884_anat"          # outside ds004884
DERIVATIVES_DIR = DEST / "derivatives"
LESION_MASK_DIR = DERIVATIVES_DIR / "lesion_mask"
ATLAS_DIR   = PROJECT_DIR / "atlases"

# Create folders
DEST.mkdir(parents=True, exist_ok=True)
DERIVATIVES_DIR.mkdir(parents=True, exist_ok=True)
LESION_MASK_DIR.mkdir(parents=True, exist_ok=True)
ATLAS_DIR.mkdir(parents=True, exist_ok=True)

# Export for bash / nilearn
os.environ["RAW_DATASET"]  = str(RAW_DATASET)
os.environ["DEST"]         = str(DEST)
os.environ["DERIVATIVES_DIR"] = str(DERIVATIVES_DIR)
os.environ["LESION_MASK_DIR"] = str(LESION_MASK_DIR)
os.environ["ATLAS_DIR"]    = str(ATLAS_DIR)
os.environ["NILEARN_DATA"] = str(ATLAS_DIR)

print("Config:")
print("  NOTEBOOK_DIR :", NOTEBOOK_DIR.resolve())
print("  RAW_DATASET  :", RAW_DATASET.resolve())
print("  DEST         :", DEST.resolve())
print("  DERIVATIVES_DIR :", DERIVATIVES_DIR.resolve())
print("  LESION_MASK_DIR :", LESION_MASK_DIR.resolve())
print("  ATLAS_DIR    :", ATLAS_DIR.resolve())

Config:
  NOTEBOOK_DIR : /neurodesktop-storage/LINDA-STUFF
  RAW_DATASET  : /neurodesktop-storage/LINDA-STUFF/ds004884
  DEST         : /neurodesktop-storage/LINDA-STUFF/ds004884_anat
  DERIVATIVES_DIR : /neurodesktop-storage/LINDA-STUFF/ds004884_anat/derivatives
  LESION_MASK_DIR : /neurodesktop-storage/LINDA-STUFF/ds004884_anat/derivatives/lesion_mask
  ATLAS_DIR    : /neurodesktop-storage/LINDA-STUFF/atlases


### Load atlases
In this workflow, we need Atlas to calculate the overlap between the lesion and the atlas regions. The following cell downloads Harvard-Oxford, AAL, Destrieux and Schaefer atlases.

In [5]:
# ============================================================
# LOCAL ATLAS LOADER
# - Stores atlases next to this notebook
# - Avoids ~/.nilearn_data
# - Robust to SSL issues for AAL
# ============================================================

print("📁 Atlas directory set to:")
print("   ", ATLAS_DIR.resolve())

# ------------------------------------------------------------
# 1. Helper class for consistent atlas interface
# ------------------------------------------------------------
class SimpleAtlas:
    def __init__(self, maps, labels):
        self.maps = maps
        self.labels = labels

# ------------------------------------------------------------
# 2. Fetch standard atlases
# ------------------------------------------------------------
print("\nFetching atlases...\n" + "-" * 50)

# Harvard–Oxford
print("• Harvard–Oxford (cortical)")
ho_atlas = datasets.fetch_atlas_harvard_oxford(
    "cort-maxprob-thr25-2mm",
    data_dir=str(ATLAS_DIR),
)
print(f"  ✓ {len(ho_atlas.labels)} regions")

# Destrieux
print("• Destrieux (2009)")
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    destrieux_atlas = datasets.fetch_atlas_destrieux_2009(
        data_dir=str(ATLAS_DIR)
    )
print(f"  ✓ {len(destrieux_atlas.labels)} regions")

# Schaefer 400
print("• Schaefer 400 (7 networks)")
schaefer_atlas = datasets.fetch_atlas_schaefer_2018(
    n_rois=400,
    yeo_networks=7,
    data_dir=str(ATLAS_DIR),
)
print(f"  ✓ {len(schaefer_atlas.labels)} regions")

# AAL
print("• AAL (SPM12)")

try:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)
        aal_obj = datasets.fetch_atlas_aal(
            version="SPM12",
            data_dir=str(ATLAS_DIR),
        )
    aal_img = nib.load(aal_obj.maps)
    aal_atlas = SimpleAtlas(aal_img, aal_obj.labels)
    print(f"  ✓ {len(aal_atlas.labels)} regions (nilearn fetch)")

except Exception as e:
    print("  ⚠ nilearn fetch failed, loading AAL from local mirror")

    import requests
    import xml.etree.ElementTree as ET

    AAL_DIR = ATLAS_DIR / "aal_mirror"
    AAL_DIR.mkdir(exist_ok=True)

    AAL_NII = AAL_DIR / "AAL.nii.gz"
    AAL_XML = AAL_DIR / "AAL.xml"

    def download(url, dest):
        if dest.exists():
            return
        r = requests.get(url, stream=True, timeout=60)
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)

    BASE = "https://raw.githubusercontent.com/Neurita/std_brains/master/atlases/aal_SPM12/aal/atlas"
    download(f"{BASE}/AAL.nii.gz", AAL_NII)
    download(f"{BASE}/AAL.xml", AAL_XML)

    aal_img = nib.load(str(AAL_NII))

    # Parse labels from XML
    labels = ["Background"]
    try:
        root = ET.parse(AAL_XML).getroot()
        for elem in root.iter():
            if elem.text:
                txt = elem.text.strip()
                if txt and txt.isalpha():
                    labels.append(txt)
    except Exception:
        pass

    aal_atlas = SimpleAtlas(aal_img, labels)
    print(f"  ✓ {len(aal_atlas.labels)} regions (mirror)")

# ------------------------------------------------------------
# 3. Final atlas registry
# ------------------------------------------------------------
ALL_ATLASES = {
    "HarvardOxford": ho_atlas,
    "AAL": aal_atlas,
    "Destrieux": destrieux_atlas,
    "Schaefer400": schaefer_atlas,
}

print("\n All atlases ready")
print("Available atlases:", list(ALL_ATLASES.keys()))


📁 Atlas directory set to:
    /neurodesktop-storage/LINDA-STUFF/atlases

Fetching atlases...
--------------------------------------------------
• Harvard–Oxford (cortical)


[fetch_atlas_harvard_oxford] Dataset found in /neurodesktop-storage/LINDA-STUFF/atlases/fsl

  ✓ 49 regions
• Destrieux (2009)


[fetch_atlas_destrieux_2009] Dataset found in /neurodesktop-storage/LINDA-STUFF/atlases/destrieux_2009

  ✓ 151 regions
• Schaefer 400 (7 networks)


...

  ✓ 401 regions
• AAL (SPM12)


[fetch_atlas_aal] Dataset found in /neurodesktop-storage/LINDA-STUFF/atlases/aal_SPM12

[fetch_atlas_aal] Downloading data from https://www.gin.cnrs.fr/AAL_files/aal_for_SPM12.tar.gz ...

[fetch_atlas_aal] Error while fetching file aal_for_SPM12.tar.gz; dataset fetching aborted.

  ⚠ nilearn fetch failed, loading AAL from local mirror
  ✓ 5 regions (mirror)

 All atlases ready
Available atlases: ['HarvardOxford', 'AAL', 'Destrieux', 'Schaefer400']


## Data preparation

### Datalad

Load 10 anatomical images (T1w as that is what LINDA uses) from the ARC dataset, containing individuals with chronic aphasia. Because some individuals have more than 1 anatomical scan, we are limiting the download to the first session of each subject. The anatomical images are stored in the `ds004884_anat` folder.

In [6]:
%%bash
set -euo pipefail
shopt -s nullglob

echo "RAW_DATASET=$RAW_DATASET"
echo "DEST=$DEST"

# -------------------------------
# Install dataset into $RAW_DATASET if missing
# -------------------------------
if [ ! -d "$RAW_DATASET" ]; then
    datalad install https://github.com/OpenNeuroDatasets/ds004884.git
    cd "$RAW_DATASET"
    git checkout 1.0.2
else
    cd "$RAW_DATASET"
fi

# -------------------------------
# Prepare folder for extracted anatomical images
# -------------------------------
mkdir -p "$DEST"

# -------------------------------
# Loop through subjects M2040–M2049
# -------------------------------
for sub in {40..49}; do
  subid=sub-M20${sub}

  echo "----------------------------------------"
  echo "Processing $subid"

  # Skip if T1w already exists in destination
  if ls "$DEST"/"$subid"/ses-*/anat/*_T1w.nii.gz &>/dev/null; then
    echo "✅ T1w already present for $subid → skipping"
    continue
  fi

  # Find first anat folder
  folder=$(find "$subid" -type d -path "*/ses-*/anat" | head -n 1 || true)

  if [ -z "$folder" ]; then
    echo "⚠️  No anat folder found for $subid"
    continue
  fi

  echo "⬇️  Downloading $folder"
  datalad get "$folder"

  # Copy T1w images + JSON
  for t1img in "$folder"/*_T1w.nii.gz; do
    json_file="${t1img%.nii.gz}.json"

    rsync -avL --relative "$t1img" "$DEST/"

    if [ -f "$json_file" ]; then
      rsync -avL --relative "$json_file" "$DEST/"
    fi
  done

  echo "✅ Finished $subid"
done


RAW_DATASET=/neurodesktop-storage/LINDA-STUFF/ds004884
DEST=/neurodesktop-storage/LINDA-STUFF/ds004884_anat
----------------------------------------
Processing sub-M2040
✅ T1w already present for sub-M2040 → skipping
----------------------------------------
Processing sub-M2041
✅ T1w already present for sub-M2041 → skipping
----------------------------------------
Processing sub-M2042
✅ T1w already present for sub-M2042 → skipping
----------------------------------------
Processing sub-M2043
✅ T1w already present for sub-M2043 → skipping
----------------------------------------
Processing sub-M2044
✅ T1w already present for sub-M2044 → skipping
----------------------------------------
Processing sub-M2045
✅ T1w already present for sub-M2045 → skipping
----------------------------------------
Processing sub-M2046
✅ T1w already present for sub-M2046 → skipping
----------------------------------------
Processing sub-M2047
✅ T1w already present for sub-M2047 → skipping
--------------------

In [7]:
%%bash
set -euo pipefail
shopt -s nullglob

input_base="$DEST"

# Check for anat directories and T1w files
echo "=== Looking for anat dirs and T1w files ==="
for subj_dir in "$input_base"/sub-*; do
    echo "Checking $subj_dir"

    # Look for anat directories at any level
    find "$subj_dir" -type d -name "anat" 2>/dev/null

    # Look for T1w files
    find "$subj_dir" -name "*_T1w.nii.gz" 2>/dev/null
done


=== Looking for anat dirs and T1w files ===
Checking /neurodesktop-storage/LINDA-STUFF/ds004884_anat/sub-M2040
/neurodesktop-storage/LINDA-STUFF/ds004884_anat/sub-M2040/ses-842/anat
/neurodesktop-storage/LINDA-STUFF/ds004884_anat/sub-M2040/ses-842/anat/sub-M2040_ses-842_acq-tfl3p2_run-2_T1w.nii.gz
Checking /neurodesktop-storage/LINDA-STUFF/ds004884_anat/sub-M2041
/neurodesktop-storage/LINDA-STUFF/ds004884_anat/sub-M2041/ses-872/anat
/neurodesktop-storage/LINDA-STUFF/ds004884_anat/sub-M2041/ses-872/anat/sub-M2041_ses-872_acq-tfl3p2_run-3_T1w.nii.gz
Checking /neurodesktop-storage/LINDA-STUFF/ds004884_anat/sub-M2042
/neurodesktop-storage/LINDA-STUFF/ds004884_anat/sub-M2042/ses-1953/anat
/neurodesktop-storage/LINDA-STUFF/ds004884_anat/sub-M2042/ses-1953/anat/sub-M2042_ses-1953_acq-tfl3p2_run-7_T1w.nii.gz
Checking /neurodesktop-storage/LINDA-STUFF/ds004884_anat/sub-M2043
/neurodesktop-storage/LINDA-STUFF/ds004884_anat/sub-M2043/ses-183/anat
/neurodesktop-storage/LINDA-STUFF/ds004884_anat/su

## Lesion segmentation

Create the derivatives location

### Single participant

for a single participant, we use the command 'linda_predict.sh "path-to-T1w-image"'


In [8]:
%%bash

set -euo pipefail

# ------------------------------------------------------------
# Input T1w image (any project, any location)
# ------------------------------------------------------------
INPUT_T1="/home/jovyan/anat/sub-M042_ses-1953_acq-tfl3p2_run-7_T1w.nii.gz" #replace with whatever file you want to segment using LINDA

# Validate input exists
if [ ! -f "$INPUT_T1" ]; then
  echo "❌ INPUT_T1 does not exist:"
  echo "   $INPUT_T1"
  exit 1
fi

# ------------------------------------------------------------
# Extract subject + session (prefer from path; fallback to filename)
# ------------------------------------------------------------
SUBJECT=$(echo "$INPUT_T1" | sed -n 's|.*/\(sub-[^/]*\)/.*|\1|p')
SESSION=$(echo "$INPUT_T1" | sed -n 's|.*/\(ses-[^/]*\)/.*|\1|p')

if [[ -z "$SUBJECT" || -z "$SESSION" ]]; then
  BASENAME=$(basename "$INPUT_T1")
  [[ -z "$SUBJECT" ]] && SUBJECT=$(echo "$BASENAME" | sed -n 's/.*\(sub-[^_]*\).*/\1/p')
  [[ -z "$SESSION" ]] && SESSION=$(echo "$BASENAME" | sed -n 's/.*\(ses-[^_]*\).*/\1/p')
fi

if [[ -z "$SUBJECT" || -z "$SESSION" ]]; then
  echo "❌ Could not extract sub-/ses- from INPUT_T1."
  echo "   Expected BIDS-like path or filename containing sub-XXX and ses-YYY."
  exit 1
fi

echo "Subject: $SUBJECT"
echo "Session: $SESSION"

# ------------------------------------------------------------
# Decide output root (strict + predictable)
# ------------------------------------------------------------
OUTPUT_ROOT=""

# 1) Use config derivatives ONLY if input is inside DEST
if [[ -n "${DEST:-}" && -n "${LESION_MASK_DIR:-}" ]]; then
  case "$INPUT_T1" in
    "$DEST"/*)
      OUTPUT_ROOT="$LESION_MASK_DIR"
      echo "📁 Using LESION_MASK_DIR from config (INPUT_T1 is under DEST):"
      echo "   $OUTPUT_ROOT"
      ;;
  esac
fi

# 2) If INPUT_T1 lives inside a /sub-.../ tree, infer project root and write derivatives there
if [[ -z "$OUTPUT_ROOT" ]]; then
  PROJECT_ROOT=$(echo "$INPUT_T1" | sed -n 's|\(.*\)/sub-[^/]\+/.*|\1|p')
  if [[ -n "$PROJECT_ROOT" ]]; then
    OUTPUT_ROOT="${PROJECT_ROOT}/derivatives/lesion_mask"
    echo "📁 Project root inferred from INPUT_T1:"
    echo "   $PROJECT_ROOT"
    echo "📁 Writing derivatives to:"
    echo "   $OUTPUT_ROOT"
  fi
fi

# 3) Final fallback: write next to the input file (never use config in this case)
if [[ -z "$OUTPUT_ROOT" ]]; then
  INPUT_DIR=$(dirname "$INPUT_T1")
  OUTPUT_ROOT="${INPUT_DIR}/derivatives/lesion_mask"
  echo "📁 No BIDS folder structure detected; writing derivatives next to INPUT_T1:"
  echo "   $OUTPUT_ROOT"
fi

# ------------------------------------------------------------
# Expected LINDA output path
# ------------------------------------------------------------
OUT_DIR="${OUTPUT_ROOT}/${SUBJECT}/${SESSION}/anat/linda"
mkdir -p "$OUT_DIR"

LINDA_OUTPUT="${OUT_DIR}/Lesion_in_MNI.nii.gz"

# ------------------------------------------------------------
# Run LINDA only if output missing
# ------------------------------------------------------------
if [[ -f "$LINDA_OUTPUT" ]]; then
  echo "✅ Existing LINDA output found:"
  echo "   $LINDA_OUTPUT"
  echo "➡️  Skipping LINDA."
else
  echo "➡️  Running LINDA on:"
  echo "   $INPUT_T1"
  echo "   Output dir: $OUT_DIR"
  echo ""

    echo "=== Current Memory State ==="
    free -h
    sync
    echo ""

  # Use the 2-arg form if your linda wrapper supports it.
  linda_predict.sh "$INPUT_T1" "$OUT_DIR"
fi

Subject: sub-M042
Session: ses-1953
📁 No BIDS folder structure detected; writing derivatives next to INPUT_T1:
   /home/jovyan/anat/derivatives/lesion_mask
➡️  Running LINDA on:
   /home/jovyan/anat/sub-M042_ses-1953_acq-tfl3p2_run-7_T1w.nii.gz
   Output dir: /home/jovyan/anat/derivatives/lesion_mask/sub-M042/ses-1953/anat/linda

=== Current Memory State ===
               total        used        free      shared  buff/cache   available
Mem:            23Gi       2.4Gi        19Gi        26Mi       1.5Gi        21Gi
Swap:          4.0Gi          0B       4.0Gi

Starting Linda prediction...
Input file: /home/jovyan/anat/sub-M042_ses-1953_acq-tfl3p2_run-7_T1w.nii.gz
Output directory: /home/jovyan/anat/derivatives/lesion_mask/sub-M042/ses-1953/anat/linda
Executing Linda prediction...
Running Linda prediction on: /home/jovyan/anat/sub-M042_ses-1953_acq-tfl3p2_run-7_T1w.nii.gz 

01:42 Starting LINDA v0.5.1
01:42 Using existing folder: /home/jovyan/anat/linda
01:42 
LINDA segmentation alrea

### Group of participants to be run as a batch

for multiple participant, looping in head folder where all sub-* are located, identified as input_base

In [9]:
%%bash

set -uo pipefail
shopt -s nullglob

# ------------------------------------------------------------
# Folder that contains sub-* directories
# ------------------------------------------------------------
INPUT_BASE="/home/jovyan/neurodesktop-storage/LINDA-STUFF/ds004884_anat"

# ------------------------------------------------------------
# Project root = whatever is before /sub-*
# (If INPUT_BASE itself contains sub-*, it *is* the project root)
# ------------------------------------------------------------
PROJECT_ROOT="$INPUT_BASE"
DERIV_ROOT="${PROJECT_ROOT}/derivatives/lesion_mask"
mkdir -p "$DERIV_ROOT"

echo "📁 INPUT_BASE:   $INPUT_BASE"
echo "📁 DERIV_ROOT:   $DERIV_ROOT"
echo ""

echo "=== Current Memory State ==="
free -h
sync
echo ""

# ------------------------------------------------------------
# Loop through all subjects / sessions / T1w images
# ------------------------------------------------------------
for subj_dir in "${INPUT_BASE}"/sub-*; do
  [ -d "$subj_dir" ] || continue

  for anat_dir in "${subj_dir}"/ses-*/anat; do
    [ -d "$anat_dir" ] || continue

    for t1img in "${anat_dir}"/*_T1w.nii.gz; do
      [ -e "$t1img" ] || continue

      # ----------------------------------------
      # Extract subject and session from filename
      # ----------------------------------------
      fname=$(basename "$t1img")
      subject=$(echo "$fname" | sed -n 's/.*\(sub-[^_]*\).*/\1/p')
      session=$(echo "$fname" | sed -n 's/.*\(ses-[^_]*\).*/\1/p')

      if [[ -z "$subject" || -z "$session" ]]; then
        echo "⚠️  Skipping (cannot parse sub/ses): $fname"
        continue
      fi

      # ----------------------------------------
      # Output directory mirrors BIDS structure
      # ----------------------------------------
      rel_path="${anat_dir#${INPUT_BASE}/}"   # sub-*/ses-*/anat
      out_dir="${DERIV_ROOT}/${rel_path}"
      mkdir -p "$out_dir"

      # ----------------------------------------
      # Expected LINDA output for *this* T1w
      # ----------------------------------------
      expected="${out_dir}/linda/Lesion_in_MNI.nii.gz"

      echo "-----------------------------------------"
      echo "Subject:  $subject"
      echo "Session:  $session"
      echo "Input:    $t1img"
      echo "Expected: $expected"
      echo "-----------------------------------------"

      # ----------------------------------------
      # Per-file existence check
      # ----------------------------------------
          if [[ -f "$expected" && -s "$expected" ]]; then
        echo "✅ Lesion mask already exists → skipping"
      else
        echo "➡️  Lesion mask not found → running LINDA"
    
        # Allow LINDA to fail without killing the loop
        linda_predict.sh "$t1img" "$out_dir" || true
    
        if [[ -f "$expected" && -s "$expected" ]]; then
          echo "✅ LINDA produced output"
        else
          echo "❌ LINDA finished but no output found"
        fi
      fi

      echo ""
      sleep 1
    done
  done
done


📁 INPUT_BASE:   /home/jovyan/neurodesktop-storage/LINDA-STUFF/ds004884_anat
📁 DERIV_ROOT:   /home/jovyan/neurodesktop-storage/LINDA-STUFF/ds004884_anat/derivatives/lesion_mask

=== Current Memory State ===
               total        used        free      shared  buff/cache   available
Mem:            23Gi       2.2Gi        19Gi        26Mi       2.2Gi        21Gi
Swap:          4.0Gi          0B       4.0Gi

-----------------------------------------
Subject:  sub-M2040
Session:  ses-842
Input:    /home/jovyan/neurodesktop-storage/LINDA-STUFF/ds004884_anat/sub-M2040/ses-842/anat/sub-M2040_ses-842_acq-tfl3p2_run-2_T1w.nii.gz
Expected: /home/jovyan/neurodesktop-storage/LINDA-STUFF/ds004884_anat/derivatives/lesion_mask/sub-M2040/ses-842/anat/linda/Lesion_in_MNI.nii.gz
-----------------------------------------
✅ Lesion mask already exists → skipping

-----------------------------------------
Subject:  sub-M2041
Session:  ses-872
Input:    /home/jovyan/neurodesktop-storage/LINDA-STUFF/ds00

### Relocate the LINDA outputs

move all the linda folders to the derivatives folder to comply with BIDS

In [10]:
%%bash

set -euo pipefail
shopt -s nullglob

input_base="ds004884_anat"
derivatives_base="ds004884_anat/derivatives/lesion_mask"

for subj_dir in "$input_base"/sub-*; do
    [ -d "$subj_dir" ] || continue
    subj_name=$(basename "$subj_dir")
    
    for ses_dir in "$subj_dir"/ses-*; do
        [ -d "$ses_dir" ] || continue
        ses_name=$(basename "$ses_dir")
        
        # Source LINDA directory (inside raw/anat)
        linda_src="$ses_dir/anat/linda"

        # Destination LINDA directory (inside derivatives)
        linda_dest="$derivatives_base/$subj_name/$ses_name/anat/linda"

        # --------------------------------------------
        # Check conditions
        # --------------------------------------------
        if [[ ! -d "$linda_src" ]]; then
            # Nothing to move
            continue
        fi

        if [[ -d "$linda_dest" ]]; then
            echo "⚠️  LINDA already exists in derivatives for $subj_name/$ses_name → skipping"
            continue
        fi

        # --------------------------------------------
        # Move LINDA output
        # --------------------------------------------
        mkdir -p "$(dirname "$linda_dest")"

        echo "➡️  Moving LINDA for $subj_name/$ses_name"
        echo "    from: $linda_src"
        echo "    to:   $linda_dest"

        mv "$linda_src" "$linda_dest"

        echo "    ✓ Done"
    done
done

echo ""
echo "✅ Finished moving LINDA outputs to derivatives/"


⚠️  LINDA already exists in derivatives for sub-M2040/ses-842 → skipping

✅ Finished moving LINDA outputs to derivatives/


confirm all files are there

In [11]:
%%bash
# Check for mask files
input_base="ds004884_anat/derivatives/lesion_mask/"

echo "=== Looking for normalised lesion mask files ==="
for subj_dir in "$input_base"/sub-*; do
    # Look for normalised lesion mask
    find "$subj_dir" -name "Lesion_in_MNI.nii.gz" 2>/dev/null
done

=== Looking for normalised lesion mask files ===
ds004884_anat/derivatives/lesion_mask//sub-M2040/ses-842/anat/linda/Lesion_in_MNI.nii.gz
ds004884_anat/derivatives/lesion_mask//sub-M2041/ses-872/anat/linda/Lesion_in_MNI.nii.gz
ds004884_anat/derivatives/lesion_mask//sub-M2042/ses-1953/anat/linda/Lesion_in_MNI.nii.gz
ds004884_anat/derivatives/lesion_mask//sub-M2043/ses-183/anat/linda/Lesion_in_MNI.nii.gz
ds004884_anat/derivatives/lesion_mask//sub-M2044/ses-3122/anat/linda/Lesion_in_MNI.nii.gz
ds004884_anat/derivatives/lesion_mask//sub-M2045/ses-381/anat/linda/Lesion_in_MNI.nii.gz
ds004884_anat/derivatives/lesion_mask//sub-M2046/ses-2499/anat/linda/Lesion_in_MNI.nii.gz
ds004884_anat/derivatives/lesion_mask//sub-M2047/ses-553/anat/linda/Lesion_in_MNI.nii.gz
ds004884_anat/derivatives/lesion_mask//sub-M2048/ses-215/anat/linda/Lesion_in_MNI.nii.gz
ds004884_anat/derivatives/lesion_mask//sub-M2049/ses-2530/anat/linda/Lesion_in_MNI.nii.gz


## Results

### QC of LINDA outputs

#### Few participants

if only a few participants, you can view individually using this first block for skull stripping

In [12]:
from ipyniivue import NiiVue
import os

# Set up paths
input_base = "ds004884_anat"
derivatives_base = "ds004884_anat/derivatives/lesion_mask"

subjects = [d for d in os.listdir(derivatives_base) if d.startswith('sub-')]

# Example: View one subject's skull stripping QC
subject = subjects[3]  # Change index to view different subjects
ses_dir = os.listdir(f"{derivatives_base}/{subject}")[0]

# T1w is still in the original location
t1_anat_dir = f"{input_base}/{subject}/{ses_dir}/anat"
# LINDA outputs are now in derivatives
linda_dir = f"{derivatives_base}/{subject}/{ses_dir}/anat/linda"

# Find the T1w file
t1_file = [f for f in os.listdir(t1_anat_dir) if f.endswith('_T1w.nii.gz')][0]

# LINDA outputs
brain_file = f"{linda_dir}/N4corrected_Brain.nii.gz"
brain_mask = f"{linda_dir}/BrainMask.nii.gz"

# View original T1 with brain extraction
nv = NiiVue()
nv.load_volumes([
    {"path": f"{t1_anat_dir}/{t1_file}"},
    {"path": brain_file, "colormap": "red", "opacity": 0.5}
])
nv

this second block for lesion mask QC

In [13]:
from ipyniivue import NiiVue
import os

# Set up paths
input_base = "ds004884_anat"
derivatives_base = "ds004884_anat/derivatives/lesion_mask"

subjects = [d for d in os.listdir(derivatives_base) if d.startswith('sub-')]

# Example: View one subject's lesion mask
subject = subjects[3]  # Change index to view different subjects
ses_dir = os.listdir(f"{derivatives_base}/{subject}")[0]

# LINDA outputs in derivatives
linda_dir = f"{derivatives_base}/{subject}/{ses_dir}/anat/linda"

# View brain with lesion overlay
nv = NiiVue() 
nv.load_volumes([
    {"path": f"{linda_dir}/N4corrected_Brain.nii.gz"},
    {"path": f"{linda_dir}/Prediction3_native.nii.gz", "colormap": "inferno", "opacity": 0.7}
])
nv

in MNI space

In [14]:
# View lesion in MNI space
nv = NiiVue()
nv.load_volumes([
    {"path": f"{linda_dir}/Subject_in_MNI.nii.gz"},
    {"path": f"{linda_dir}/Lesion_in_MNI.nii.gz", "colormap": "inferno", "opacity": 0.7}
])
nv

#### Group viewer

alternatively, this block allows you to view all subjects in a slider, which is more convenient for larger datasets

In [15]:
from ipyniivue import NiiVue
import os
from IPython.display import display, HTML
import ipywidgets as widgets

input_base = "ds004884_anat"
derivatives_base = "ds004884_anat/derivatives/lesion_mask"

# Get list of all subjects
all_subjects = []
for subj_dir in sorted([d for d in os.listdir(derivatives_base) if d.startswith('sub-')]):
    for ses_dir in os.listdir(f"{derivatives_base}/{subj_dir}"):
        if not ses_dir.startswith('ses-'):
            continue
        linda_dir = f"{derivatives_base}/{subj_dir}/{ses_dir}/anat/linda"
        if os.path.exists(linda_dir):
            all_subjects.append((subj_dir, ses_dir))

output = widgets.Output()

def show_subject(index):
    with output:
        output.clear_output(wait=True)
        if 0 <= index < len(all_subjects):
            subj_dir, ses_dir = all_subjects[index]
            
            # T1w is in original location
            t1_anat_dir = f"{input_base}/{subj_dir}/{ses_dir}/anat"
            # LINDA outputs in derivatives
            linda_dir = f"{derivatives_base}/{subj_dir}/{ses_dir}/anat/linda"
            
            display(HTML(f"<h3>Subject {index + 1}/{len(all_subjects)}: {subj_dir} - {ses_dir}</h3>"))
            
            # Skull stripping QC
            display(HTML("<h4>Skull Stripping QC</h4>"))
            nv1 = NiiVue()
            nv1.load_volumes([
                {"path": f"{linda_dir}/N4corrected.nii.gz"},
                {"path": f"{linda_dir}/BrainMask.nii.gz", "colormap": "red", "opacity": 0.3}
            ])
            display(nv1)
            
            # Lesion mask QC - Native Space
            display(HTML("<h4>Lesion Mask QC - Native Space</h4>"))
            nv2 = NiiVue()
            nv2.load_volumes([
                {"path": f"{linda_dir}/N4corrected_Brain.nii.gz"},
                {"path": f"{linda_dir}/Prediction3_native.nii.gz", "colormap": "inferno", "opacity": 0.7}
            ])
            display(nv2)
            
            # Lesion mask QC - MNI Space
            display(HTML("<h4>Lesion Mask QC - MNI Space</h4>"))
            nv3 = NiiVue()
            nv3.load_volumes([
                {"path": f"{linda_dir}/Subject_in_MNI.nii.gz"},
                {"path": f"{linda_dir}/Lesion_in_MNI.nii.gz", "colormap": "inferno", "opacity": 0.7}
            ])
            display(nv3)

# Create navigation slider
slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(all_subjects)-1,
    step=1,
    description='Subject:',
    continuous_update=False
)

widgets.interact(show_subject, index=slider)
display(output)

interactive(children=(IntSlider(value=0, continuous_update=False, description='Subject:', max=9), Output()), _…

Output()

## Calculate overlap - all atlases

First we load all the atlases 

In [16]:
!sudo apt-get update
!sudo apt-get install -y ca-certificates
!sudo update-ca-certificates

Get:1 https://deb.nodesource.com/node_20.x nodistro InRelease [12.1 kB]
Get:2 https://packages.microsoft.com/repos/code stable InRelease [3590 B]      
Get:3 https://packages.microsoft.com/repos/code stable/main arm64 Packages [25.8 kB]
Get:4 https://deb.nodesource.com/node_20.x nodistro/main arm64 Packages [14.3 kB]
Get:5 https://ppa.launchpadcontent.net/apptainer/ppa/ubuntu noble InRelease [17.8 kB]
Ign:6 http://cvmrepo.s3.cern.ch/cvmrepo/apt noble-prod InRelease               
Get:7 http://ports.ubuntu.com/ubuntu-ports noble InRelease [256 kB] 
Get:8 https://ppa.launchpadcontent.net/mozillateam/ppa/ubuntu noble InRelease [24.4 kB]
Get:9 http://cvmrepo.s3.cern.ch/cvmrepo/apt noble-prod Release [7191 B]        
Get:10 http://cvmrepo.s3.cern.ch/cvmrepo/apt noble-prod Release.gpg [833 B]    
Get:11 https://ppa.launchpadcontent.net/nextcloud-devs/client/ubuntu noble InRelease [24.1 kB]
Get:12 https://ppa.launchpadcontent.net/apptainer/ppa/ubuntu noble/main arm64 Packages [739 B]
Get:13 h

Then we can calculate the overlap between the lesion and the atlas regions for all participants. 
Options for atlases are: ATLAS_NAME = "Schaefer400"  # Options: "HarvardOxford", "AAL", "Destrieux", "Schaefer400"
or 


In [17]:
import nibabel as nib
import numpy as np
import pandas as pd
from nilearn import datasets, image
import os
import warnings
from ipyniivue import NiiVue
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ============================================
# ATLAS SELECTION - CHANGE THIS
# ============================================
PROCESS_ALL_ATLASES = True  # Set to True to process all atlases, False to process only one
ATLAS_NAME = "HarvardOxford"  # Only used if PROCESS_ALL_ATLASES = False
# ============================================

derivatives_base = "ds004884_anat/derivatives/lesion_mask"

# Dictionary of available atlases
def load_atlas(atlas_name):
    """Load the specified atlas and return atlas object with standardized structure"""
    
    print(f"Loading {atlas_name} atlas...")
    
    if atlas_name == "HarvardOxford":
        atlas_obj = datasets.fetch_atlas_harvard_oxford('cort-maxprob-thr25-2mm')
        atlas_img = atlas_obj.maps
        atlas_data = atlas_img.get_fdata()
        atlas_labels = atlas_obj.labels
        
    elif atlas_name == "AAL":
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", DeprecationWarning)
            atlas_obj = datasets.fetch_atlas_aal(version='SPM12')
        atlas_img = nib.load(atlas_obj.maps)
        atlas_data = atlas_img.get_fdata()
        atlas_labels = atlas_obj.labels
        
    elif atlas_name == "Destrieux":
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            atlas_obj = datasets.fetch_atlas_destrieux_2009()
        atlas_img = nib.load(atlas_obj.maps)
        atlas_data = atlas_img.get_fdata()
        atlas_labels = atlas_obj.labels
        
    elif atlas_name == "Schaefer400":
        atlas_obj = datasets.fetch_atlas_schaefer_2018(n_rois=400, yeo_networks=7)
        atlas_img = nib.load(atlas_obj.maps)
        atlas_data = atlas_img.get_fdata()
        atlas_labels = atlas_obj.labels
        
    else:
        raise ValueError(f"Unknown atlas: {atlas_name}. Choose from: HarvardOxford, AAL, Destrieux, Schaefer400")
    
    print(f"✓ {atlas_name} atlas loaded: {len(atlas_labels)} regions")
    
    return {
        'name': atlas_name,
        'img': atlas_img,
        'data': atlas_data,
        'labels': atlas_labels
    }

def calculate_overlap(lesion_path, atlas_img, atlas_data, atlas_labels):
    """Calculate percentage overlap between lesion and atlas regions"""
    
    if not os.path.exists(lesion_path):
        print(f"Lesion file not found: {lesion_path}")
        return {}, 0
    
    lesion_img = nib.load(lesion_path)
    
    lesion_resampled = image.resample_to_img(
        lesion_img, 
        atlas_img, 
        interpolation='nearest',
        force_resample=True,
        copy_header=True
    )
    lesion_data = lesion_resampled.get_fdata()
    
    lesion_binary = (lesion_data > 0.5).astype(int)
    total_lesion_voxels = np.sum(lesion_binary)
    
    if total_lesion_voxels == 0:
        print("Warning: No lesion voxels found")
        return {}, 0
    
    overlaps = {}
    
    for region_idx in np.unique(atlas_data):
        if region_idx == 0:
            continue
        
        region_idx = int(region_idx)
        region_mask = (atlas_data == region_idx)
        total_region_voxels = np.sum(region_mask)
        overlap_voxels = np.sum(lesion_binary & region_mask)
        
        if overlap_voxels > 0:
            lesion_in_roi_percent = (overlap_voxels / total_lesion_voxels) * 100
            roi_coverage_percent = (overlap_voxels / total_region_voxels) * 100
            
            if region_idx < len(atlas_labels):
                region_name = atlas_labels[region_idx]
            else:
                region_name = f"Region_{region_idx}"
            
            overlaps[region_name] = {
                'lesion_in_roi_percent': lesion_in_roi_percent,
                'roi_coverage_percent': roi_coverage_percent,
                'overlap_voxels': overlap_voxels,
                'total_region_voxels': total_region_voxels
            }
    
    return overlaps, total_lesion_voxels

# Determine which atlases to process
if PROCESS_ALL_ATLASES:
    atlases_to_process = ["HarvardOxford", "AAL", "Destrieux", "Schaefer400"]
    print("Processing all atlases...")
else:
    atlases_to_process = [ATLAS_NAME]
    print(f"Processing {ATLAS_NAME} atlas only...")

print("="*70)

# Process each atlas
for current_atlas_name in atlases_to_process:
    print(f"\n{'='*70}")
    print(f"PROCESSING WITH {current_atlas_name} ATLAS")
    print(f"{'='*70}")
    
    # Load the atlas
    atlas = load_atlas(current_atlas_name)
    atlas_img = atlas['img']
    atlas_data = atlas['data']
    atlas_labels = atlas['labels']
    
    # Save atlas to file for visualization
    atlas_path = f"{derivatives_base}/{current_atlas_name}_atlas.nii.gz"
    if not os.path.exists(atlas_path):
        print(f"Saving {current_atlas_name} atlas to file...")
        nib.save(atlas_img, atlas_path)
        print(f"✓ Atlas saved to {atlas_path}")
    
    # Process all subjects with this atlas
    print(f"\nProcessing all subjects with {current_atlas_name} atlas...")
    print("-"*70)
    
    all_results = []
    
    for subj_dir in sorted([d for d in os.listdir(derivatives_base) if d.startswith('sub-')]):
        for ses_dir in os.listdir(f"{derivatives_base}/{subj_dir}"):
            if not ses_dir.startswith('ses-'):
                continue
            
            lesion_path = f"{derivatives_base}/{subj_dir}/{ses_dir}/anat/linda/Lesion_in_MNI.nii.gz"
            
            if not os.path.exists(lesion_path):
                continue
            
            print(f"Processing {subj_dir} - {ses_dir}...", end=" ")
            
            overlaps, total_voxels = calculate_overlap(lesion_path, atlas_img, atlas_data, atlas_labels)
            
            print(f"✓ ({total_voxels} voxels, {len(overlaps)} regions)")
            
            for region, data in overlaps.items():
                all_results.append({
                    'subject': subj_dir,
                    'session': ses_dir,
                    'atlas': current_atlas_name,
                    'region': region,
                    'lesion_in_roi_percent': data['lesion_in_roi_percent'],
                    'roi_coverage_percent': data['roi_coverage_percent'],
                    'overlap_voxels': data['overlap_voxels'],
                    'total_region_voxels': data['total_region_voxels'],
                    'total_lesion_voxels': total_voxels
                })
    
    # Create DataFrame
    df = pd.DataFrame(all_results)
    
    # Save to CSV
    output_file = f"{derivatives_base}/lesion_atlas_overlap_{current_atlas_name}.csv"
    df.to_csv(output_file, index=False)
    print(f"\n✓ Results saved to: {output_file}")
    print(f"  Total subjects: {df['subject'].nunique()}")
    print(f"  Total regions: {df['region'].nunique()}")

print("\n" + "="*70)
print("ALL PROCESSING COMPLETE!")
print("="*70)

if PROCESS_ALL_ATLASES:
    print("\nGenerated files:")
    for atlas_name in atlases_to_process:
        print(f"  - lesion_atlas_overlap_{atlas_name}.csv")

Processing all atlases...

PROCESSING WITH HarvardOxford ATLAS
Loading HarvardOxford atlas...


[fetch_atlas_harvard_oxford] Dataset found in /neurodesktop-storage/LINDA-STUFF/atlases/fsl

✓ HarvardOxford atlas loaded: 49 regions

Processing all subjects with HarvardOxford atlas...
----------------------------------------------------------------------
Processing sub-M2040 - ses-842... ✓ (23316 voxels, 25 regions)
Processing sub-M2041 - ses-872... ✓ (14749 voxels, 22 regions)
Processing sub-M2042 - ses-1953... ✓ (22297 voxels, 23 regions)
Processing sub-M2043 - ses-183... ✓ (1 voxels, 0 regions)
Processing sub-M2044 - ses-3122... ✓ (17141 voxels, 31 regions)
Processing sub-M2045 - ses-381... ✓ (19829 voxels, 27 regions)
Processing sub-M2046 - ses-2499... ✓ (17115 voxels, 28 regions)
Processing sub-M2047 - ses-553... ✓ (10450 voxels, 18 regions)
Processing sub-M2048 - ses-215... ✓ (7150 voxels, 18 regions)
Processing sub-M2049 - ses-2530... ✓ (2854 voxels, 6 regions)

✓ Results saved to: ds004884_anat/derivatives/lesion_mask/lesion_atlas_overlap_HarvardOxford.csv
  Total subjects: 9
  Total regions: 42

PROCESSING WITH AAL ATLAS
Loading AAL atlas...


[fetch_atlas_aal] Dataset found in /neurodesktop-storage/LINDA-STUFF/atlases/aal_SPM12

[fetch_atlas_aal] Downloading data from https://www.gin.cnrs.fr/AAL_files/aal_for_SPM12.tar.gz ...

[fetch_atlas_aal] Error while fetching file aal_for_SPM12.tar.gz; dataset fetching aborted.

SSLError: HTTPSConnectionPool(host='www.gin.cnrs.fr', port=443): Max retries exceeded with url: /AAL_files/aal_for_SPM12.tar.gz (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)')))

We can visualise these results for each participant by modifying the 'SUBJECT =' field in the cell below. This will allow you to generate a report for a specific subject and session.

In [ ]:
def calculate_overlap(lesion_path, atlas_img, atlas_data, atlas_labels):
    """Calculate percentage overlap between lesion and atlas regions"""
    
    if not os.path.exists(lesion_path):
        print(f"Lesion file not found: {lesion_path}")
        return {}, 0
    
    # Load lesion in MNI space
    lesion_img = nib.load(lesion_path)
    
    # Resample lesion to match atlas if needed
    lesion_resampled = image.resample_to_img(
        lesion_img, 
        atlas_img, 
        interpolation='nearest',
        force_resample=True,
        copy_header=True
    )
    lesion_data = lesion_resampled.get_fdata()
    
    # Binarize lesion (threshold at 0.5 for probability maps)
    lesion_binary = (lesion_data > 0.5).astype(int)
    
    # Calculate total lesion volume
    total_lesion_voxels = np.sum(lesion_binary)
    
    if total_lesion_voxels == 0:
        print("Warning: No lesion voxels found")
        return {}, 0
    
    # Calculate overlaps with each region
    overlaps = {}
    
    for region_idx in np.unique(atlas_data):
        if region_idx == 0:  # Skip background
            continue
        
        region_idx = int(region_idx)
        region_mask = (atlas_data == region_idx)
        
        # Calculate total voxels in this region
        total_region_voxels = np.sum(region_mask)
        
        # Calculate overlap
        overlap_voxels = np.sum(lesion_binary & region_mask)
        
        if overlap_voxels > 0:
            # Percentage of lesion in this region
            lesion_in_roi_percent = (overlap_voxels / total_lesion_voxels) * 100
            
            # Percentage of ROI covered by lesion
            roi_coverage_percent = (overlap_voxels / total_region_voxels) * 100
            
            # Get region name
            if region_idx < len(atlas_labels):
                region_name = atlas_labels[region_idx]
            else:
                region_name = f"Region_{region_idx}"
            
            overlaps[region_name] = {
                'lesion_in_roi_percent': lesion_in_roi_percent,
                'roi_coverage_percent': roi_coverage_percent,
                'overlap_voxels': overlap_voxels,
                'total_region_voxels': total_region_voxels
            }
    
    return overlaps, total_lesion_voxels

# ============================================
# ATLAS SELECTION - CHANGE THIS
# ============================================
ATLAS_NAME = "Schaefer400"  # Options: "HarvardOxford", "AAL", "Destrieux", "Schaefer400"
# ============================================

# ============================================
# SUBJECT SELECTION - CHANGE THIS
# ============================================
SUBJECT = "sub-M2040"  # ← CHANGE THIS
SESSION = None         # ← Leave as None to auto-detect, or specify like "ses-842"
# ============================================

derivatives_base = "ds004884_anat/derivatives/lesion_mask"

# Load the selected atlas
atlas = load_atlas(ATLAS_NAME)
atlas_img = atlas['img']
atlas_data = atlas['data']
atlas_labels = atlas['labels']

# Auto-detect session if not specified
if SESSION is None:
    subject_path = f"{derivatives_base}/{SUBJECT}"
    if os.path.exists(subject_path):
        sessions = [d for d in os.listdir(subject_path) if d.startswith('ses-')]
        if sessions:
            SESSION = sessions[0]  # Use first session
            print(f"Auto-detected session: {SESSION}")

if SESSION:
    test_lesion = f"{derivatives_base}/{SUBJECT}/{SESSION}/anat/linda/Lesion_in_MNI.nii.gz"
    
    if os.path.exists(test_lesion):
        overlaps, total_voxels = calculate_overlap(test_lesion, atlas_img, atlas_data, atlas_labels)
        print(f"\n{'='*80}")
        print(f"LESION ATLAS OVERLAP: {SUBJECT} - {SESSION}")
        print(f"Atlas: {ATLAS_NAME}")
        print(f"{'='*80}")
        print(f"\nTotal lesion volume: {total_voxels} voxels")
        print(f"\nRegions affected (sorted by % of lesion in ROI):")
        print(f"{'-'*80}")
        print(f"{'Region':<45s} {'Lesion in ROI':>14s} {'ROI Coverage':>12s} {'Voxels':>8s}")
        print(f"{'-'*80}")
        
        for region, data in sorted(overlaps.items(), key=lambda x: x[1]['lesion_in_roi_percent'], reverse=True):
            print(f"{region:<45s} "
                  f"{data['lesion_in_roi_percent']:>13.1f}% "
                  f"{data['roi_coverage_percent']:>11.1f}% "
                  f"{data['overlap_voxels']:>8.0f}")
    else:
        print(f"ERROR: Lesion file not found for {SUBJECT} - {SESSION}")
else:
    print("Cannot generate report - no valid session")

The results can be saved for an individual participant.

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
from datetime import datetime

def create_subject_report_pdf(subj_dir, ses_dir, df, atlas_name, derivatives_base, output_dir=None):
    """Create a PDF report for one subject"""
    
    subject_data = df[(df['subject'] == subj_dir) & (df['session'] == ses_dir)]
    
    if len(subject_data) == 0:
        print(f"No data found for {subj_dir} - {ses_dir}")
        return None
    
    subject_data = subject_data.sort_values('lesion_in_roi_percent', ascending=False)
    
    # Set output directory
    if output_dir is None:
        output_dir = f"{derivatives_base}/reports"
    os.makedirs(output_dir, exist_ok=True)
    
    # Create PDF filename
    pdf_filename = f"{output_dir}/{subj_dir}_{ses_dir}_{atlas_name}_lesion_report.pdf"
    
    with PdfPages(pdf_filename) as pdf:
        # Page 1: Summary and top regions
        fig = plt.figure(figsize=(8.5, 11))
        fig.suptitle(f'Lesion Anatomical Report\n{subj_dir} - {ses_dir}', 
                     fontsize=16, fontweight='bold')
        
        # Add summary text
        total_voxels = subject_data['total_lesion_voxels'].iloc[0]
        summary_text = f"""
Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}
Atlas: {atlas_name}

Total Lesion Volume: {total_voxels} voxels
Number of Regions Affected: {len(subject_data)}

Top 15 Affected Regions (by % of lesion in ROI):
"""
        
        plt.text(0.1, 0.85, summary_text, transform=fig.transFigure, 
                fontsize=11, verticalalignment='top', family='monospace')
        
        # Add top 15 regions table
        table_text = f"{'Rank':<5s}{'Region':<40s}{'Lesion in ROI':>12s}{'ROI Coverage':>12s}\n"
        table_text += "-" * 69 + "\n"
        for idx, (_, row) in enumerate(subject_data.head(15).iterrows(), 1):
            table_text += f"{idx:<5d}{row['region']:<40s}{row['lesion_in_roi_percent']:>11.1f}%{row['roi_coverage_percent']:>11.1f}%\n"
        
        plt.text(0.1, 0.55, table_text, transform=fig.transFigure,
                fontsize=9, verticalalignment='top', family='monospace')
        
        plt.axis('off')
        pdf.savefig(fig, bbox_inches='tight')
        plt.close()
        
        # Page 2+: Complete regional distribution table
        rows_per_page = 30
        num_pages = (len(subject_data) + rows_per_page - 1) // rows_per_page
        
        for page in range(num_pages):
            fig = plt.figure(figsize=(8.5, 11))
            
            start_idx = page * rows_per_page
            end_idx = min((page + 1) * rows_per_page, len(subject_data))
            page_data = subject_data.iloc[start_idx:end_idx]
            
            title = f'Complete Regional Distribution (Page {page + 1}/{num_pages})'
            plt.text(0.5, 0.95, title, transform=fig.transFigure,
                    fontsize=12, fontweight='bold', ha='center')
            
            # Create table text
            table_text = f"{'Rank':<6s}{'Region':<40s}{'Lesion in ROI':>14s}{'ROI Coverage':>12s}{'Voxels':>8s}\n"
            table_text += "-" * 80 + "\n"
            
            for idx, (_, row) in enumerate(page_data.iterrows(), start_idx + 1):
                table_text += f"{idx:<6d}{row['region']:<40s}{row['lesion_in_roi_percent']:>13.1f}%{row['roi_coverage_percent']:>11.1f}%{row['overlap_voxels']:>8.0f}\n"
            
            plt.text(0.1, 0.88, table_text, transform=fig.transFigure,
                    fontsize=8, verticalalignment='top', family='monospace')
            
            plt.axis('off')
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
        
        # Add metadata
        d = pdf.infodict()
        d['Title'] = f'Lesion Report: {subj_dir} - {ses_dir} ({atlas_name})'
        d['Author'] = 'LINDA Lesion Analysis'
        d['Subject'] = f'Lesion-Atlas Overlap Analysis - {atlas_name} Atlas'
        d['CreationDate'] = datetime.now()
    
    print(f"✓ PDF report saved to: {pdf_filename}")
    return pdf_filename

# ============================================
# ATLAS SELECTION FOR PDF GENERATION
# ============================================
PDF_ATLAS_NAME = "Destrieux"  # Options: "HarvardOxford", "AAL", "Destrieux", "Schaefer400"
# ============================================

# ============================================
# PDF GENERATION OPTIONS
# ============================================
GENERATE_ALL_SUBJECTS = True  # True = all subjects, False = single subject
SUBJECT = "sub-M2040"  # Only used if GENERATE_ALL_SUBJECTS = False
SESSION = None         # Only used if GENERATE_ALL_SUBJECTS = False
# ============================================

derivatives_base = "ds004884_anat/derivatives/lesion_mask"

# Load the data for the selected atlas
csv_file = f"{derivatives_base}/lesion_atlas_overlap_{PDF_ATLAS_NAME}.csv"

if not os.path.exists(csv_file):
    print(f"ERROR: CSV file not found: {csv_file}")
    print(f"Please run the atlas processing first with ATLAS_NAME = '{PDF_ATLAS_NAME}'")
else:
    df = pd.read_csv(csv_file)
    
    if GENERATE_ALL_SUBJECTS:
        # Generate PDFs for all subjects
        print(f"Generating PDF reports for all subjects using {PDF_ATLAS_NAME} atlas...")
        print("="*70)
        
        for subj_dir in sorted(df['subject'].unique()):
            for ses_dir in df[df['subject'] == subj_dir]['session'].unique():
                print(f"Generating report for {subj_dir} - {ses_dir}...")
                create_subject_report_pdf(subj_dir, ses_dir, df, PDF_ATLAS_NAME, derivatives_base)
        
        print("\n✓ All PDF reports generated!")
        print(f"Reports saved in: {derivatives_base}/reports/")
        
    else:
        # Generate PDF for single subject
        # Auto-detect session if not specified
        if SESSION is None:
            subject_path = f"{derivatives_base}/{SUBJECT}"
            if os.path.exists(subject_path):
                sessions = [d for d in os.listdir(subject_path) if d.startswith('ses-')]
                if sessions:
                    SESSION = sessions[0]
                    print(f"Auto-detected session: {SESSION}")
        
        if SESSION:
            print(f"Generating PDF report for {SUBJECT} - {SESSION} using {PDF_ATLAS_NAME} atlas...")
            pdf_file = create_subject_report_pdf(SUBJECT, SESSION, df, PDF_ATLAS_NAME, derivatives_base)
        else:
            print("Cannot generate report - no valid session")

In [ ]:
from ipyniivue import NiiVue
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import nibabel as nib
import os
import pandas as pd

derivatives_base = "ds004884_anat/derivatives/lesion_mask"

# ============================================
# Load all available atlases and dataframes
# ============================================
available_atlases = []
atlas_data = {}

for atlas_name in ["HarvardOxford", "AAL", "Destrieux", "Schaefer400"]:
    csv_file = f"{derivatives_base}/lesion_atlas_overlap_{atlas_name}.csv"
    atlas_file = f"{derivatives_base}/{atlas_name}_atlas.nii.gz"
    
    if os.path.exists(csv_file) and os.path.exists(atlas_file):
        available_atlases.append(atlas_name)
        atlas_data[atlas_name] = {
            'df': pd.read_csv(csv_file),
            'path': atlas_file
        }
        print(f"✓ Loaded {atlas_name} atlas and data")
    else:
        print(f"⊘ Skipping {atlas_name} - missing files")

if not available_atlases:
    print("\nERROR: No atlas data found!")
    print("Please run the atlas processing first.")
else:
    print(f"\n✓ {len(available_atlases)} atlases available")

def create_subject_report_with_viz(subj_dir, ses_dir, atlas_name, derivatives_base):
    """Create a detailed report with visualization for one subject"""
    
    df = atlas_data[atlas_name]['df']
    atlas_path = atlas_data[atlas_name]['path']
    
    subject_data = df[(df['subject'] == subj_dir) & (df['session'] == ses_dir)]
    
    if len(subject_data) == 0:
        print(f"No data found for {subj_dir} - {ses_dir} with {atlas_name} atlas")
        return None
    
    subject_data = subject_data.sort_values('lesion_in_roi_percent', ascending=False)
    
    # Header
    display(HTML(f"<h2>LESION ANATOMICAL REPORT: {subj_dir} - {ses_dir}</h2>"))
    display(HTML(f"<p><strong>Atlas:</strong> {atlas_name}</p>"))
    
    total_voxels = subject_data['total_lesion_voxels'].iloc[0]
    display(HTML(f"<p><strong>Total lesion volume:</strong> {total_voxels} voxels</p>"))
    display(HTML(f"<p><strong>Number of regions affected:</strong> {len(subject_data)}</p>"))
    
    # Visualization
    display(HTML("<h3>Lesion Visualization (MNI Space)</h3>"))
    lesion_path = f"{derivatives_base}/{subj_dir}/{ses_dir}/anat/linda/Lesion_in_MNI.nii.gz"
    subject_mni_path = f"{derivatives_base}/{subj_dir}/{ses_dir}/anat/linda/Subject_in_MNI.nii.gz"
    
    nv = NiiVue()
    nv.load_volumes([
        {"path": subject_mni_path},
        {"path": atlas_path, "colormap": "viridis", "opacity": 0.4},
        {"path": lesion_path, "colormap": "red", "opacity": 0.4}
    ])
    display(nv)
    
    # Region table
    display(HTML("<h3>Regional Distribution</h3>"))
    
    table_html = """
    <table style="width:100%; border-collapse: collapse; font-family: monospace; font-size: 12px;">
        <tr style="background-color: #2c3e50; color: white; border-bottom: 2px solid #34495e;">
            <th style="text-align: left; padding: 12px 8px; font-weight: bold;">Region</th>
            <th style="text-align: right; padding: 12px 8px; font-weight: bold;">Lesion in ROI</th>
            <th style="text-align: right; padding: 12px 8px; font-weight: bold;">ROI Coverage</th>
            <th style="text-align: right; padding: 12px 8px; font-weight: bold;">Voxels</th>
        </tr>
    """
    
    for idx, row in subject_data.iterrows():
        table_html += f"""
        <tr style="border-bottom: 1px solid #ddd;">
            <td style="padding: 8px;">{row['region']}</td>
            <td style="text-align: right; padding: 8px;">{row['lesion_in_roi_percent']:.1f}%</td>
            <td style="text-align: right; padding: 8px;">{row['roi_coverage_percent']:.1f}%</td>
            <td style="text-align: right; padding: 8px;">{row['overlap_voxels']:.0f}</td>
        </tr>
        """
    
    table_html += "</table>"
    display(HTML(table_html))
    
    return subject_data

if available_atlases:
    # Get all unique subject/session combinations from first available atlas
    first_atlas = available_atlases[0]
    df = atlas_data[first_atlas]['df']
    subject_sessions = df[['subject', 'session']].drop_duplicates().reset_index(drop=True)
    subject_labels = [f"{row['subject']} - {row['session']}" for _, row in subject_sessions.iterrows()]
    
    output = widgets.Output()
    
    def show_report(subject_label, atlas_name):
        with output:
            output.clear_output(wait=True)
            # Find the subject and session from the label
            index = subject_labels.index(subject_label)
            subj = subject_sessions.iloc[index]['subject']
            ses = subject_sessions.iloc[index]['session']
            create_subject_report_with_viz(subj, ses, atlas_name, derivatives_base)
    
    # Create dropdown for subjects
    subject_dropdown = widgets.Dropdown(
        options=subject_labels,
        description='Subject:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    
    # Create radio buttons for atlas selection
    atlas_radio = widgets.RadioButtons(
        options=available_atlases,
        description='Atlas:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )
    
    # Display controls
    display(HTML("<h2>Interactive Lesion Report Viewer</h2>"))
    
    # Arrange controls side by side
    controls = widgets.HBox([subject_dropdown, atlas_radio])
    
    # Interactive display
    widgets.interact(show_report, subject_label=subject_dropdown, atlas_name=atlas_radio)
    display(output)